In [21]:
pip install transformers datasets peft accelerate tqdm

In [22]:
from datasets import load_dataset
dataset = load_dataset("sciq")

In [23]:
train_data = dataset["train"].select(range(2000))
test_data = dataset["validation"].select(range(10))

In [24]:
def format_example(example):
  return f"Question:{example['question']}\nAnswer:{example['correct_answer']}"

In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "distilgpt2"
tokenizer =AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
pip install --upgrade torchao

In [27]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [28]:
def tokenize(example):
    text = format_example(example)
    return tokenizer(text, truncation=True, padding="max_length", max_length=128)

tokenizer.pad_token = tokenizer.eos_token
train_data = train_data.map(tokenize, remove_columns=['question', 'distractor1', 'distractor2', 'distractor3', 'correct_answer', 'support'])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [29]:
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm
from transformers import DataCollatorWithPadding

# Initialize the data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(train_data, batch_size=8, shuffle=True, collate_fn=data_collator)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

for epoch in range(2):
    loop = tqdm(train_loader, leave=True)

    for batch in loop:
        # DataCollatorWithPadding returns tensors directly
        # The batch dictionary will contain 'input_ids' and 'attention_mask' as tensors
        inputs = batch["input_ids"].to(device)
        # For causal language modeling, labels are usually the same as input_ids
        labels = inputs.clone()

        outputs = model(inputs, labels=labels)

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        loop.set_description(f"Epoch {epoch}")
        loop.set_postfix(loss=loss.item())

Epoch 1: 100%|██████████| 250/250 [00:31<00:00,  7.93it/s, loss=0.957]


In [32]:
def generate_answer(question):
    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [33]:
results = []

for example in test_data:
    q = example["question"]
    real = example["correct_answer"]

    pred = generate_answer(q)

    results.append({
        "question": q,
        "real_answer": real,
        "generated": pred
    })

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [34]:
import json

with open("lora_results.json", "w") as f:
    json.dump(results, f, indent=4)

In [35]:
for r in results:
    print("Q:", r["question"])
    print("Real:", r["real_answer"])
    print("Model:", r["generated"])
    print("-"*50)

Q: Who proposed the theory of evolution by natural selection?
Real: darwin
Model: Question: Who proposed the theory of evolution by natural selection?
Answer:
The theory of evolution is the theory of evolution.
--------------------------------------------------
Q: Each specific polypeptide has a unique linear sequence of which acids?
Real: amino
Model: Question: Each specific polypeptide has a unique linear sequence of which acids?
Answer:
--------------------------------------------------
Q: A frameshift mutation is a deletion or insertion of one or more of what that changes the reading frame of the base sequence?
Real: nucleotides
Model: Question: A frameshift mutation is a deletion or insertion of one or more of what that changes the reading frame of the base sequence?
Answer:
--------------------------------------------------
Q: What is an area of land called that is wet for all or part of the year?
Real: wetland
Model: Question: What is an area of land called that is wet for all o